# **TaniMol: 02 - Fingerprint Generation**

This notebook loads the preprocessed dataset from `01_preprocessing` and generates molecular fingerprints for each compound. The fingerprints are binary bit vectors that encode structural features and will be used to compute pairwise Tanimoto similarity in the next step.

**Input:** `data/processed/cleaned_activities.csv`  
**Fingerprint types:** Morgan (ECFP4), MACCS keys, RDKit topological  
**Output:** fingerprint arrays ready for similarity computation

In [1]:
import pandas as pd
import numpy as np

from src.config import OUTPUT_PATH
from src.fingerprints import (
    mol_from_smiles,
    generate_morgan_fp,
    generate_maccs_fp,
    generate_rdkit_fp,
    add_fingerprints,
)

### **1. Load Preprocessed Data**

Load the cleaned dataset produced by `01_preprocessing`. Each row is a unique (target, molecule) pair with a standardized SMILES and pIC50 value.

In [2]:
df = pd.read_csv(OUTPUT_PATH)

### **2. Generate Morgan Fingerprints (ECFP4)**

Morgan fingerprints with radius=2 (ECFP4) capture circular substructures around each atom up to 2 bonds away. This is the standard fingerprint for similarity-based analysis in drug discovery. Each molecule becomes a 2048-bit binary vector.

In [3]:
morgan_fps = add_fingerprints(df, fp_type="morgan")

100%|██████████| 11012/11012 [00:01<00:00, 7258.30it/s]


### **3. Generate MACCS Keys**

MACCS keys use 166 predefined structural patterns (e.g. "contains aromatic ring", "has nitrogen"). Less granular than Morgan but useful for comparison — different fingerprint types can produce different similarity rankings.

In [4]:
maccs_fps = add_fingerprints(df, fp_type="maccs")

100%|██████████| 11012/11012 [00:07<00:00, 1394.20it/s]


### **4. Generate RDKit Topological Fingerprints**

RDKit fingerprints encode topological paths (linear sequences of bonds) of various lengths. Path-based rather than circular — captures different structural information than Morgan.

In [5]:
rdkit_fps = add_fingerprints(df, fp_type="rdkit")

100%|██████████| 11012/11012 [00:15<00:00, 714.04it/s]


### **5. Quick Sanity Check**

Verify that the fingerprints look reasonable: check bit density (fraction of \"on\" bits). Typical ECFP4 density is ~1-5% for drug-like molecules. For MACCS this number is around 33%.

In [6]:
for name, fps in [("Morgan", morgan_fps), ("MACCS", maccs_fps), ("RDKit", rdkit_fps)]:
    valid = [fp for fp in fps if fp is not None]
    avg_density = np.mean([fp.mean() for fp in valid]) * 100
    none_count = len(fps) - len(valid)
    print(f"{name}: Density: {avg_density:.2f}%, Failed: {none_count}/{len(fps)}")

Morgan: Density: 2.71%, Failed: 0/11012
MACCS: Density: 33.96%, Failed: 0/11012
RDKit: Density: 52.79%, Failed: 0/11012
